# Bayesian Network from Scratch: Rooftop Beehive Health 🐝

A **Bayesian Network (BN)** is made of two parts:
1. A **DAG** (directed acyclic graph): each node is a random variable, and each arrow means "directly influences".
2. A **Conditional Probability Table (CPT)** for every node: `P(node | parents)`.

Together they define the full joint distribution with the **chain rule**:

$$P(X_1, \dots, X_n) = \prod_i P(X_i \mid \text{Parents}(X_i))$$

### The story (synthetic dataset)
A city beekeeper keeps hives on building rooftops. At every inspection they record:

| Variable | Values | Meaning |
|---|---|---|
| `Season` | Spring / Summer / Monsoon / Winter | Time of year |
| `Pesticide_Nearby` | Yes / No | Was a nearby park or garden sprayed recently? |
| `Flower_Bloom` | Low / High | How many flowers are available for foraging |
| `Mite_Load` | Low / High | Level of Varroa mites in the hive |
| `Colony_Health` | Weak / Strong | Overall strength of the colony |
| `Honey_Yield` | Low / Medium / High | Honey harvested |
| `Swarming` | Yes / No | Did the colony try to swarm (split and leave)? |

In this notebook we will:
1. Define the network structure
2. **Generate** a synthetic dataset from hidden "true" probabilities
3. **Learn** the CPTs back from the data, using only counting
4. Run **inference** (ask probability questions) with enumeration

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Network structure

Each node maps to the list of its parents. `ORDER` is a topological order (parents always come before their children).

In [ ]:
STATES = {
    "Season":           ["Spring", "Summer", "Monsoon", "Winter"],
    "Pesticide_Nearby": ["Yes", "No"],
    "Flower_Bloom":     ["Low", "High"],
    "Mite_Load":        ["Low", "High"],
    "Colony_Health":    ["Weak", "Strong"],
    "Honey_Yield":      ["Low", "Medium", "High"],
    "Swarming":         ["Yes", "No"],
}

parents = {
    "Season":           [],
    "Pesticide_Nearby": [],
    "Flower_Bloom":     ["Season"],
    "Mite_Load":        ["Season"],
    "Colony_Health":    ["Mite_Load", "Pesticide_Nearby"],
    "Honey_Yield":      ["Flower_Bloom", "Colony_Health"],
    "Swarming":         ["Colony_Health", "Flower_Bloom"],
}

ORDER = list(parents)  # already written in topological order

In [ ]:
# Draw the DAG with plain matplotlib (fixed positions, no extra libraries)
pos = {
    "Season": (1.5, 4), "Pesticide_Nearby": (4.2, 4),
    "Flower_Bloom": (0.7, 2.6), "Mite_Load": (2.6, 3),
    "Colony_Health": (3.8, 2), "Honey_Yield": (2.3, 1.1), "Swarming": (2.3, 0),
}
role = {"Season": "cause", "Pesticide_Nearby": "cause",
        "Flower_Bloom": "middle", "Mite_Load": "middle", "Colony_Health": "middle",
        "Honey_Yield": "outcome", "Swarming": "outcome"}
colors = {"cause": "#cfe3ff", "middle": "#ffe9a8", "outcome": "#c8f0c8"}

fig, ax = plt.subplots(figsize=(9, 6))
for child, pas in parents.items():
    for p in pas:
        ax.annotate("", xy=pos[child], xytext=pos[p],
                    arrowprops=dict(arrowstyle="-|>", color="gray", lw=1.5,
                                    shrinkA=20, shrinkB=22, mutation_scale=15))
for node, (x, y) in pos.items():
    ax.text(x, y, node, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.5", fc=colors[role[node]], ec="dimgray"))
for r, c in colors.items():
    ax.scatter([], [], s=150, marker="s", color=c, edgecolor="dimgray", label=r)
ax.legend(title="Node role", loc="lower right", frameon=False)
ax.set_xlim(-0.2, 5.2); ax.set_ylim(-0.5, 4.5); ax.axis("off")
ax.set_title("Rooftop Beehive Bayesian Network")
plt.show()

## 2. Generate the dataset

We write down some **"true" CPTs** by hand. They are only used to *create* the data. Later we pretend we never saw them and learn the probabilities back from the data alone.

Each CPT is a dict: `(parent values) -> {state: probability}`.

In [ ]:
TRUE_CPTS = {
    "Season": {(): {"Spring": 0.25, "Summer": 0.25, "Monsoon": 0.30, "Winter": 0.20}},
    "Pesticide_Nearby": {(): {"Yes": 0.3, "No": 0.7}},
    "Flower_Bloom": {
        ("Spring",):  {"Low": 0.15, "High": 0.85},
        ("Summer",):  {"Low": 0.40, "High": 0.60},
        ("Monsoon",): {"Low": 0.70, "High": 0.30},   # rain keeps bees inside
        ("Winter",):  {"Low": 0.90, "High": 0.10},
    },
    "Mite_Load": {
        ("Spring",):  {"Low": 0.80, "High": 0.20},
        ("Summer",):  {"Low": 0.50, "High": 0.50},
        ("Monsoon",): {"Low": 0.35, "High": 0.65},   # humidity helps mites
        ("Winter",):  {"Low": 0.70, "High": 0.30},
    },
    "Colony_Health": {   # parents: (Mite_Load, Pesticide_Nearby)
        ("Low", "No"):   {"Weak": 0.10, "Strong": 0.90},
        ("Low", "Yes"):  {"Weak": 0.45, "Strong": 0.55},
        ("High", "No"):  {"Weak": 0.60, "Strong": 0.40},
        ("High", "Yes"): {"Weak": 0.90, "Strong": 0.10},
    },
    "Honey_Yield": {     # parents: (Flower_Bloom, Colony_Health)
        ("Low", "Weak"):    {"Low": 0.85, "Medium": 0.13, "High": 0.02},
        ("Low", "Strong"):  {"Low": 0.40, "Medium": 0.45, "High": 0.15},
        ("High", "Weak"):   {"Low": 0.50, "Medium": 0.40, "High": 0.10},
        ("High", "Strong"): {"Low": 0.05, "Medium": 0.35, "High": 0.60},
    },
    "Swarming": {        # parents: (Colony_Health, Flower_Bloom)
        ("Weak", "Low"):    {"Yes": 0.05, "No": 0.95},
        ("Weak", "High"):   {"Yes": 0.10, "No": 0.90},
        ("Strong", "Low"):  {"Yes": 0.15, "No": 0.85},
        ("Strong", "High"): {"Yes": 0.55, "No": 0.45},  # strong + lots of food = swarm
    },
}

In [ ]:
def sample_row():
    """Sample one hive inspection by walking the nodes in topological order."""
    row = {}
    for node in ORDER:
        key = tuple(row[p] for p in parents[node])
        probs = TRUE_CPTS[node][key]
        row[node] = str(np.random.choice(list(probs), p=list(probs.values())))
    return row

df = pd.DataFrame([sample_row() for _ in range(3000)])
df.to_csv("rooftop_beehives.csv", index=False)
print(df.shape)
df.head(10)

In [ ]:
# Quick look at how each variable is distributed
for col in df.columns:
    print(col, df[col].value_counts().to_dict())

## 3. Learn the CPTs from the data

To learn `P(node | parents)` we just **count**:

$$P(X = x \mid \text{pa}) = \frac{\text{count}(X = x, \text{pa}) + 1}{\text{count}(\text{pa}) + k}$$

The `+1` / `+k` is **Laplace smoothing** (k = number of states). It makes sure no probability is ever exactly 0.

In [ ]:
def learn_cpt(df, node):
    """Estimate P(node | parents) by counting rows (with Laplace smoothing)."""
    pa, cpt = parents[node], {}
    for key in itertools.product(*[STATES[p] for p in pa]):
        mask = np.ones(len(df), dtype=bool)
        for p, v in zip(pa, key):
            mask &= (df[p] == v)
        counts = df.loc[mask, node].value_counts()
        total = mask.sum() + len(STATES[node])
        cpt[key] = {s: (counts.get(s, 0) + 1) / total for s in STATES[node]}
    return cpt

cpts = {node: learn_cpt(df, node) for node in ORDER}

In [ ]:
def show_cpt(node, cpt):
    """Pretty-print a CPT as a DataFrame."""
    table = pd.DataFrame(cpt).T.round(3)
    if parents[node]:
        table.index.names = parents[node]
    print(f"P({node} | {', '.join(parents[node]) or '-'})")
    display(table)

for node in ORDER:
    show_cpt(node, cpts[node])

## 4. Joint probability (chain rule)

The probability of one *complete* assignment of all 7 variables is the product of one entry from each CPT.

In [ ]:
def joint_prob(assign, cpts=cpts):
    """P(full assignment) = product of P(node | parents) over all nodes."""
    p = 1.0
    for node in ORDER:
        key = tuple(assign[q] for q in parents[node])
        p *= cpts[node][key][assign[node]]
    return p

example = {"Season": "Spring", "Pesticide_Nearby": "No", "Flower_Bloom": "High",
           "Mite_Load": "Low", "Colony_Health": "Strong",
           "Honey_Yield": "High", "Swarming": "Yes"}
print("P(perfect spring hive that swarms) =", round(joint_prob(example), 4))

## 5. Inference by enumeration

To answer a question like `P(target | evidence)`:
1. Loop over **every** full assignment (only 4·2·2·2·2·3·2 = 768 of them).
2. Keep the ones that agree with the evidence.
3. Add up their joint probabilities, grouped by the value of the target.
4. Normalize so the result sums to 1.

This is the simplest exact inference method. It is slow for big networks but works perfectly here.

In [ ]:
def query(target, evidence={}, cpts=cpts):
    """Exact inference: P(target | evidence) by summing over all assignments."""
    result = {s: 0.0 for s in STATES[target]}
    for values in itertools.product(*[STATES[n] for n in ORDER]):
        assign = dict(zip(ORDER, values))
        if all(assign[k] == v for k, v in evidence.items()):
            result[assign[target]] += joint_prob(assign, cpts)
    total = sum(result.values())
    return pd.Series({s: p / total for s, p in result.items()}).round(3)

## 6. Asking the network questions

**Q1. With no information at all, what honey yield should we expect?**

In [ ]:
query("Honey_Yield")

**Q2. What if we know it is Monsoon season?** (Fewer flowers and more mites, so the yield should drop.)

In [ ]:
query("Honey_Yield", {"Season": "Monsoon"})

**Q3. A hive just swarmed. How healthy was the colony?**

This is reasoning *against* the arrows, from an effect back to its cause. Weak colonies rarely swarm, so seeing a swarm makes "Strong" much more likely.

In [ ]:
print("Prior:            ", query("Colony_Health").to_dict())
print("Given Swarming=Yes:", query("Colony_Health", {"Swarming": "Yes"}).to_dict())

**Q4. Explaining away.** A colony is weak. Was it mites or pesticide?

- If we only know the colony is **Weak**, high mites become more likely.
- If we *also* learn there was **pesticide** nearby, the pesticide already "explains" the weakness, so the probability of high mites goes back **down**.

In [ ]:
print("P(Mite_Load | Weak):                   ", query("Mite_Load", {"Colony_Health": "Weak"}).to_dict())
print("P(Mite_Load | Weak, Pesticide=Yes):    ", query("Mite_Load", {"Colony_Health": "Weak", "Pesticide_Nearby": "Yes"}).to_dict())
print("P(Mite_Load | Weak, Pesticide=No):     ", query("Mite_Load", {"Colony_Health": "Weak", "Pesticide_Nearby": "No"}).to_dict())

**Q5. Beekeeper's best case vs worst case: chance of a High honey yield**

In [ ]:
best  = query("Honey_Yield", {"Season": "Spring", "Pesticide_Nearby": "No"})
worst = query("Honey_Yield", {"Season": "Monsoon", "Pesticide_Nearby": "Yes"})
pd.DataFrame({"Spring, no pesticide": best, "Monsoon + pesticide": worst})

## 7. Sanity checks

1. The network's answers should be close to simple **frequencies counted from the data**.
2. The learned CPTs should be close to the hidden **true CPTs** that generated the data.
3. Running inference with the **true** CPTs should give similar answers.

In [ ]:
# 1. BN inference vs. raw frequency in the data
empirical = df[df.Season == "Monsoon"].Honey_Yield.value_counts(normalize=True)
pd.DataFrame({
    "BN (learned)": query("Honey_Yield", {"Season": "Monsoon"}),
    "BN (true CPTs)": query("Honey_Yield", {"Season": "Monsoon"}, cpts=TRUE_CPTS),
    "Data frequency": empirical.reindex(STATES["Honey_Yield"]).round(3),
})

In [ ]:
# 2. Largest gap between any learned and true probability
max_gap = max(
    abs(cpts[n][k][s] - TRUE_CPTS[n][k][s])
    for n in ORDER for k in TRUE_CPTS[n] for s in STATES[n]
)
print(f"Largest |learned - true| over all CPT entries: {max_gap:.3f}")

## Summary

- We built a **7-node Bayesian Network** for rooftop beehive health with no BN library, using only `numpy`, `pandas` and `itertools`.
- **Structure** is a `parents` dictionary. **Parameters** (CPTs) are learned by counting, with Laplace smoothing.
- **Joint probability** comes from the chain rule. **Inference** is brute-force enumeration with normalization.
- The network handles predictive reasoning (Season → Yield), diagnostic reasoning (Swarming → Health) and **explaining away** (Pesticide vs Mites).

**Ideas to extend:** add a `Weather_Today` node, try variable elimination for faster inference, or learn the structure from the data with a score such as BIC.